# Mistério em João Pessoa

Adaptação do lendário [SQL Murder Mystery](https://github.com/NUKnightLab/sql-mysteries) (Knight Lab / Northwestern University, conteúdo original sob licença CC BY-SA 4.0). Nesta adaptação você resolve tudo com pandas + MinIO. A cidade e alguns nomes/ruas viraram locais de João Pessoa.

Um assassinato foi registrado em **João Pessoa** em **15/01/2018**. A polícia recolheu 6 arquivos crus (`dados/*.csv`) e é com eles que você vai trabalhar.

## O que você recebeu

| Arquivo | Colunas | O que é |
|---|---|---|
| `ocorrencia.csv` | `data, tipo, descricao, cidade` | Boletins de Ocorrência |
| `pessoa.csv` | `id, nome, detran_id, numero_endereco, rua, cpf` | Cadastro de Pessoas |
| `detran.csv` | `id, idade, altura, cor_olhos, cor_cabelo, genero, placa, marca_veiculo, modelo_veiculo` | Cadastro do DETRAN |
| `depoimento.csv` | `pessoa_id, relato` | Depoimentos |
| `membro_academia.csv` | `id, pessoa_id, nome, data_matricula, plano` | Matrículas da Academia |
| `checkin_academia.csv` | `matricula_id, data_checkin, hora_entrada, hora_saida` | Check-ins da Academia |

## O que você entrega

1. **Fase 1 — Bronze**: os 6 CSVs publicados como tabelas bronze (`df.to_parquet(f"s3://{BUCKET}/bronze/<nome>.parquet", storage_options=STORAGE_OPTIONS)`).
2. **Fase 2 — Investigação**: livre — use pandas (`pd.read_parquet` + `merge`/filtros) para seguir as pistas até chegar a **1** suspeito.
3. **Fase 3 — Resposta final na Silver**: uma tabela `silver.resposta_caso` com sua conclusão (ver Fase 3 no final deste notebook pro formato esperado).

In [106]:
import os

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

# Conexão com o MinIO (S3-compatível) — pandas usa isso direto via s3fs
BUCKET = os.environ.get("LAKEHOUSE_S3_BUCKET", "lakehouse")
STORAGE_OPTIONS = {
    "key": os.environ.get("LAKEHOUSE_S3_ACCESS_KEY", "trilha"),
    "secret": os.environ.get("LAKEHOUSE_S3_SECRET_KEY", "trilha123"),
    "client_kwargs": {"endpoint_url": os.environ.get("LAKEHOUSE_S3_ENDPOINT", "http://minio:9000")},
}

## Fase 1 — Bronze

O padrão pra publicar qualquer CSV cru como tabela bronze é sempre o mesmo — 2 passos, só pandas:

```python
df = pd.read_csv("dados/<arquivo>.csv")                                                                     # lê o CSV cru direto do disco
df.to_parquet(f"s3://{BUCKET}/bronze/<nome_tabela>.parquet", storage_options=STORAGE_OPTIONS, index=False)  # publica: grava Parquet
```

Cada tabela vira **1 arquivo Parquet** na camada — `bronze/<nome_tabela>.parquet`, dá pra conferir pelo MinIO Console (http://localhost:9001) — os arquivos vão aparecendo em `lakehouse/bronze/`.

Um exemplo pronto (`ocorrencia`), com uma ilustração rápida de como ler de volta logo depois — daí é sua vez de fazer o mesmo padrão para as outras 5 tabelas.

In [107]:
# Exemplo pronto: ocorrencia
df_ocorrencia = pd.read_csv("dados/ocorrencia.csv")
df_ocorrencia.to_parquet(f"s3://{BUCKET}/bronze/ocorrencia.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_ocorrencia

,data,tipo,descricao,cidade
0,20180115,roubo,A Man Dressed as Spider-Man Is on a Robbery Spree,NYC
1,20180115,assassinato,Life? Dont talk to me about life.,Albany
2,20180115,assassinato,"Mama, I killed a man, put a gun against his head...",Reno
3,20180215,assassinato,REDACTED REDACTED REDACTED,João Pessoa
4,20180215,assassinato,Someone killed the guard! He took an arrow to the knee!,João Pessoa
...,...,...,...,...
1223,20180430,suborno,\n,Garden Grove
1224,20180430,fraude,‘Why not?’ said the March Hare.\n,Houma
1225,20180430,agressão,\n,Fontana
1226,20180501,agressão,"be NO mistake about it: it was neither more nor less than a pig, and she\n",Trenton


### Lendo de volta (exemplo rápido)

Pra ler uma tabela publicada é só apontar o `pd.read_parquet` pro mesmo caminho, ilustrado com a tabela que acabamos de publicar:

- `pd.read_parquet(f"s3://{BUCKET}/bronze/<tabela>.parquet", storage_options=STORAGE_OPTIONS)` lê o Parquet inteiro direto do MinIO com pandas — é o que você vai usar na Fase 2 pra ler cada uma das 6 tabelas bronze.

Pra ver o que já existe fisicamente numa camada, sem precisar ler o conteúdo nem escrever código nenhum: abra o **MinIO Console** (http://localhost:9001) e olhe os arquivos em `lakehouse/bronze/`.

In [108]:
# Lendo a tabela de volta direto do MinIO com pandas — deve ser idêntica a df_ocorrencia
pd.read_parquet(f"s3://{BUCKET}/bronze/ocorrencia.parquet", storage_options=STORAGE_OPTIONS)

,data,tipo,descricao,cidade
0,20180115,roubo,A Man Dressed as Spider-Man Is on a Robbery Spree,NYC
1,20180115,assassinato,Life? Dont talk to me about life.,Albany
2,20180115,assassinato,"Mama, I killed a man, put a gun against his head...",Reno
3,20180215,assassinato,REDACTED REDACTED REDACTED,João Pessoa
4,20180215,assassinato,Someone killed the guard! He took an arrow to the knee!,João Pessoa
...,...,...,...,...
1223,20180430,suborno,\n,Garden Grove
1224,20180430,fraude,‘Why not?’ said the March Hare.\n,Houma
1225,20180430,agressão,\n,Fontana
1226,20180501,agressão,"be NO mistake about it: it was neither more nor less than a pig, and she\n",Trenton


In [109]:
# TODO: repita o padrão para "pessoa.csv" -> bronze.pessoa
df_pessoa = pd.read_csv("dados/pessoa.csv")
df_pessoa.to_parquet(f"s3://{BUCKET}/bronze/pessoa.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_pessoa

,id,nome,detran_id,numero_endereco,rua,cpf
0,10000,Christoper Peteuil,993845,624,Bankhall Ave,747714076
1,10007,Kourtney Calderwood,861794,2791,Gustavus Blvd,477972044
2,10010,Muoi Cary,385336,741,Avenida Ministro José Américo de Almeida,828638512
3,10016,Era Moselle,431897,1987,Wood Glade St,614621061
4,10025,Trena Hornby,550890,276,Daws Hill Way,223877684
...,...,...,...,...,...,...
10006,99936,Luba Benser,274427,680,Carnage Blvd,685095054
10007,99941,Roxana Mckimley,975942,1613,Gate St,512136801
10008,99965,Cherie Zeimantz,287627,3661,The Water Ave,362877324
10009,99982,Allen Cruse,251350,3126,N Jean Dr,348734531


In [110]:
# TODO: repita o padrão para "detran.csv" -> bronze.detran
df_detran = pd.read_csv("dados/detran.csv")
df_detran.to_parquet(f"s3://{BUCKET}/bronze/detran.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_detran

,id,idade,altura,cor_olhos,cor_cabelo,genero,placa,marca_veiculo,modelo_veiculo
0,100280,72,57,castanho,ruivo,masculino,P24L4U,Acura,MDX
1,100460,63,72,castanho,castanho,feminino,XF02T6,Cadillac,SRX
2,101029,62,74,verde,verde,feminino,VKY5KR,Scion,xB
3,101198,43,54,mel,castanho,feminino,Y5NZ08,Nissan,Rogue
4,101255,18,79,azul,grisalho,feminino,5162Z1,Lexus,GS
...,...,...,...,...,...,...,...,...,...
10002,999923,19,77,mel,preto,feminino,5L0ZI4,GMC,Sierra 3500
10003,999940,71,61,verde,verde,masculino,1B8QN8,Mitsubishi,Eclipse
10004,999981,67,69,castanho,azul,feminino,1684K3,Land Rover,LR2
10005,999986,49,58,verde,grisalho,masculino,F8F64H,Lexus,LS


In [111]:
# TODO: repita o padrão para "depoimento.csv" -> bronze.depoimento
df_depoimento = pd.read_csv("dados/depoimento.csv")
df_depoimento.to_parquet(f"s3://{BUCKET}/bronze/depoimento.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_depoimento

,pessoa_id,relato
0,28508,‘I deny it!’ said the March Hare.\n
1,63713,\n
2,86208,"way, and the whole party swam to the shore.\n"
3,35267,"lessons in here? Why, there’s hardly room for YOU, and no room at all\n"
4,33856,\n
...,...,...
4986,37357,"Alice did not wish to offend the Dormouse again, so she began very\n"
4987,10206,"time,’ she said, ‘than waste it in asking riddles that have no answers.’\n"
4988,14887,"Eu ouvi um tiro e depois vi um homem saindo correndo. Ele tinha uma bolsa da ""Academia Kongo"". O número da matrícula na bolsa começava com ""48Z"". Só sócios do plano ouro têm essas bolsas. O homem entrou num carro com uma placa que continha ""H42W""."
4989,16371,"Eu vi o assassinato acontecer, e reconheci o assassino da minha academia, de quando eu estava treinando na semana passada, no dia 9 de janeiro."


In [112]:
# TODO: repita o padrão para "membro_academia.csv" -> bronze.membro_academia
df_membro_academia = pd.read_csv("dados/membro_academia.csv")
df_membro_academia.to_parquet(f"s3://{BUCKET}/bronze/membro_academia.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_membro_academia

,id,pessoa_id,nome,data_matricula,plano
0,NL318,65076,Everette Koepke,20170926,ouro
1,AOE21,39426,Noe Locascio,20171005,regular
2,2PN28,63823,Jeromy Heitschmidt,20180215,prata
3,0YJ24,80651,Waneta Wellard,20171206,ouro
4,3A08L,32858,Mei Bianchin,20170401,prata
...,...,...,...,...,...
179,2V137,41693,Wendell Dulany,20171219,prata
180,4KB72,79110,Emile Hege,20170522,regular
181,48Z7A,28819,Severino Bezerra Lima,20160305,ouro
182,48Z55,67318,Genildo Cavalcanti Farias,20160101,ouro


In [113]:
# TODO: repita o padrão para "checkin_academia.csv" -> bronze.checkin_academia
df_checkin_academia = pd.read_csv("dados/checkin_academia.csv")
df_checkin_academia.to_parquet(f"s3://{BUCKET}/bronze/checkin_academia.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_checkin_academia

,matricula_id,data_checkin,hora_entrada,hora_saida
0,NL318,20180212,329,365
1,NL318,20170811,469,920
2,NL318,20180429,506,554
3,NL318,20180128,124,759
4,NL318,20171027,418,1019
...,...,...,...,...
2698,4KB72,20170422,1016,1114
2699,4KB72,20170630,408,885
2700,48Z7A,20180109,1600,1730
2701,48Z55,20180109,1530,1700


Checagem: as 6 tabelas devem aparecer em `lakehouse/bronze/` no **MinIO Console** (http://localhost:9001).

## Fase 2 — Investigação (pandas)

A partir de agora é livre: leia as tabelas bronze direto do MinIO com `pd.read_parquet(f"s3://{BUCKET}/bronze/<tabela>.parquet", storage_options=STORAGE_OPTIONS)`, e siga as pistas com `merge`/filtros de DataFrame, do jeito que preferir.

Um roteiro sugerido (não obrigatório seguir exatamente esta ordem, mas ajuda a não se perder):

1. Pesquise a ocorrência e bus as testemunhas em `pessoa`.
2. Pesquise o `depoimento` das respectivas testemunhas.
3. Cada depoimento traz uma pista diferente — uma aponta para `membro_academia` a outra para `detran`.
4. Pesquise pela pistas, mencionadas pelas testemunhas, nas tabelas correspondentes.
5. Confirme em `checkin_academia` que o suspeito tem um check-in na academia na data que a segunda testemunha mencionou.

In [114]:
# Lendo todas as tabelas bronze direto do MinIO via pandas
ocorrencia = pd.read_parquet(f"s3://{BUCKET}/bronze/ocorrencia.parquet", storage_options=STORAGE_OPTIONS)

### Passo 1 — a ocorrência

Um assassinato foi registrado em **João Pessoa** em **15/01/2018**.

In [115]:
# TODO:

df_ocorrencia.sample(10)

,data,tipo,descricao,cidade
47,20170113,chantagem,"‘There’s no sort of use in knocking,’ said the Footman, ‘and that for\n",Fort Collins
606,20170826,furto,\n,Thornton
429,20170611,assassinato,"Be off, or I’ll kick you down stairs!’\n",Frederick
196,20170310,incêndio criminoso,\n,GreenBay
179,20170302,assassinato,"with the words ‘DRINK ME,’ but nevertheless she uncorked it and put it\n",Mobile
55,20170116,chantagem,\n,Ontario
757,20171023,contrabando,"candle is like after the candle is blown out, for she could not remember\n",Murfreesboro
974,20180115,fraude,"The Dormouse slowly opened his eyes. ‘I wasn’t asleep,’ he said in a\n",Yakima
1129,20180325,fraude,corners: next the ten courtiers; these were ornamented all over with\n,Coral Springs
1018,20180201,suborno,\n,Baton Rouge


In [116]:
df_ocorrencia.dtypes

data          int64
tipo         object
descricao    object
cidade       object
dtype: object

In [117]:
df_ocorrencia["tipo"].unique()

array(['roubo', 'assassinato', 'furto', 'fraude', 'incêndio criminoso',
       'suborno', 'agressão', 'contrabando', 'chantagem'], dtype=object)

In [118]:
df_ocorrencia["cidade"].unique()

array(['NYC', 'Albany', 'Reno', 'João Pessoa', 'Chicago', 'Seattle',
       'Long Beach', 'Oceanside', 'Gilbert', 'Nashville', 'Las Vegas',
       'Riverside', 'Canton', 'Alexandria', 'Rochester', 'Lacey',
       'McAllen', 'Seaside', 'Lewisville', 'Jackson', 'High Point',
       'Lorain', 'Savannah', 'Newark', 'Oklahoma City', 'Memphis',
       'Round Lake Beach', 'Louisville', 'Jacksonville', 'Grand Rapids',
       'North Port', 'Port St. Lucie', 'Little Rock', 'Visalia', 'Kailua',
       'Springdale', 'Lancaster', 'Melbourne', 'Paterson', 'Fort Collins',
       'Reading', 'Orange', 'Jersey City', 'Hollywood', 'Cincinnati',
       'Ontario', 'Plano', 'Tallahassee', 'Las Cruces', 'Anaheim',
       'San Bernardino', 'Charlotte', 'Wilmington', 'Denton', 'Asheville',
       'Knoxville', 'Murfreesboro', 'Houston', 'Oakland', 'Omaha',
       'Temecula', 'Layton', 'Arvada', 'Sterling Heights', 'Arlington',
       'Santa Clarita', 'Macon', 'Waterbury', 'Laredo', 'New Bedford',
       'Richmo

In [119]:
df_ocorrencia[(df_ocorrencia["tipo"] == "assassinato") & (df_ocorrencia["cidade"] == "João Pessoa") & (df_ocorrencia["data"] == 20180115)]

,data,tipo,descricao,cidade
1227,20180115,assassinato,"As imagens de segurança mostram que houve 2 testemunhas. A primeira testemunha mora na última casa da ""Avenida Ministro José Américo de Almeida"". A segunda testemunha, chamada Maria Aparecida, mora em algum lugar da ""Avenida Rui Carneiro"".",João Pessoa


In [120]:
resultado_ocorrencia = df_ocorrencia[(df_ocorrencia["tipo"] == "assassinato") & (df_ocorrencia["cidade"] == "João Pessoa") & (df_ocorrencia["data"] == 20180115)]

### Passo 2 — as testemunhas

In [121]:
# TODO: 
df_pessoa.sample(10)

,id,nome,detran_id,numero_endereco,rua,cpf
1868,27222,Ashlyn Dallis,609889,2179,Topping St,792629129
8140,83164,Isa Iberra,212240,2620,Ryon Rd,631500698
5143,56154,Wes Lawrentz,986137,2002,Sweetbirch St,954459211
4259,48307,Marquetta Chan,781583,2544,Southeast Bain St,427166816
256,12394,Marna Turberville,760432,558,Appley Rd,658012151
6801,71364,Elizbeth Englert,907936,1366,W Revere Blvd,990745937
4643,51552,Bonita Knie,764124,1726,Avenida Ministro José Américo de Almeida,597137908
8575,87103,Keisha Heimbuch,803192,1797,Borges Ranch Circle,369283022
7548,77939,Amado Liuzzi,218012,2012,Woids Dr,568761219
7724,79450,Heriberto Obremski,914543,3137,Elk Lick Way,188571249


In [122]:
df_pessoa.dtypes

id                  int64
nome               object
detran_id           int64
numero_endereco     int64
rua                object
cpf                 int64
dtype: object

In [123]:
casas_jose_americo = df_pessoa[(df_pessoa["rua"] == "Avenida Ministro José Américo de Almeida")]
casas_jose_americo.sort_values("numero_endereco", ascending=False)

,id,nome,detran_id,numero_endereco,rua,cpf
499,14887,Josenildo Pereira da Rocha,118009,4919,Avenida Ministro José Américo de Almeida,111564949
811,17729,Lasonya Wildey,439686,3824,Avenida Ministro José Américo de Almeida,917817122
4886,53890,Sophie Tiberio,957671,3755,Avenida Ministro José Américo de Almeida,442830147
7040,73368,Torie Thalmann,773862,3697,Avenida Ministro José Américo de Almeida,341559436
9609,96595,Coretta Cubie,303645,3631,Avenida Ministro José Américo de Almeida,378403829
1021,19420,Cody Schiel,890431,3524,Avenida Ministro José Américo de Almeida,947110049
9272,93509,Emmitt Aceuedo,916706,3491,Avenida Ministro José Américo de Almeida,979073160
8608,87456,Leonora Wolfsberger,215868,3483,Avenida Ministro José Américo de Almeida,565203106
2871,36378,Freddie Ellzey,267882,3449,Avenida Ministro José Américo de Almeida,474117596
4806,53076,Boris Bijou,664914,3327,Avenida Ministro José Américo de Almeida,401191868


In [124]:
df_pessoa[(df_pessoa["rua"] == "Avenida Rui Carneiro") & df_pessoa["nome"].str.contains("Maria Aparecida")]

,id,nome,detran_id,numero_endereco,rua,cpf
665,16371,Maria Aparecida Nunes,490173,103,Avenida Rui Carneiro,318771143


### Passo 3 — duas pistas, duas tabelas


In [125]:
# TODO:
df_depoimento.sample(10)

,pessoa_id,relato
109,55804,leading right into it. ‘That’s very curious!’ she thought. ‘But\n
1301,58107,"feebly stretching out one paw, trying to touch her. ‘Poor little thing!’\n"
4837,60920,\n
2396,44344,"looked at the sides of the well, and noticed that they were filled with\n"
2378,50463,Who would not give all else for two\n
44,82421,"can’t think! And oh, I wish you could see her after the birds! Why,\n"
1198,79506,"that she did not dare to laugh; and, as she could not think of anything\n"
32,23260,The Mock Turtle went on.\n
1000,44988,"as the Rabbit, and had no reason to be afraid of it.\n"
512,27190,"the moral of that is--“The more there is of mine, the less there is of\n"


In [126]:
df_depoimento.dtypes

pessoa_id     int64
relato       object
dtype: object

In [127]:
df_depoimento[df_depoimento["pessoa_id"].isin([14887, 16371])]

,pessoa_id,relato
4988,14887,"Eu ouvi um tiro e depois vi um homem saindo correndo. Ele tinha uma bolsa da ""Academia Kongo"". O número da matrícula na bolsa começava com ""48Z"". Só sócios do plano ouro têm essas bolsas. O homem entrou num carro com uma placa que continha ""H42W""."
4989,16371,"Eu vi o assassinato acontecer, e reconheci o assassino da minha academia, de quando eu estava treinando na semana passada, no dia 9 de janeiro."


In [128]:
# TODO:
df_membro_academia.sample(10)

,id,pessoa_id,nome,data_matricula,plano
59,GKZV7,30221,Jude Fairbairn,20170108,ouro
181,48Z7A,28819,Severino Bezerra Lima,20160305,ouro
87,03487,73490,Lyman Harshbarger,20170529,prata
108,7A4NF,98747,Aleen Bergmann,20170703,prata
31,82GA2,20716,Roslyn Gonzaga,20170809,ouro
121,MYQ6A,99462,Darren Yahraus,20170712,prata
125,QN47X,53753,Francisco Harcrow,20171214,regular
178,HM6U8,50106,Edgar Bamba,20170128,prata
46,2B01Y,47735,Tania Kubesh,20170417,ouro
76,OBU3O,80489,Wilma Sinstack,20170207,regular


In [129]:
df_membro_academia.dtypes

id                object
pessoa_id          int64
nome              object
data_matricula     int64
plano             object
dtype: object

In [130]:
df_membro_academia["plano"].unique()

array(['ouro', 'regular', 'prata'], dtype=object)

In [131]:
df_membro_academia[(df_membro_academia["plano"] == "ouro") & df_membro_academia["id"].str.startswith("48Z")]

,id,pessoa_id,nome,data_matricula,plano
181,48Z7A,28819,Severino Bezerra Lima,20160305,ouro
182,48Z55,67318,Genildo Cavalcanti Farias,20160101,ouro


In [132]:
suspeitos_academia = df_membro_academia[(df_membro_academia["plano"] == "ouro") & df_membro_academia["id"].str.startswith("48Z")]

In [133]:
df_detran.sample(10)

,id,idade,altura,cor_olhos,cor_cabelo,genero,placa,marca_veiculo,modelo_veiculo
5252,572512,29,60,preto,ruivo,masculino,64K766,Hyundai,Elantra
4815,538031,30,76,castanho,loiro,masculino,5U5WJ6,Mercedes-Benz,M-Class
3222,396405,22,82,azul,ruivo,feminino,68SK8A,Land Rover,LR3
1384,228647,21,74,mel,grisalho,feminino,64SV07,Maybach,62
7274,753402,46,74,verde,ruivo,masculino,G8DLTI,Toyota,Matrix
7367,763360,22,70,mel,ruivo,masculino,0XH8S1,Toyota,Sequoia
4741,531430,55,82,azul,ruivo,feminino,C5O842,GMC,Safari
4802,537252,60,64,verde,castanho,masculino,D43O7A,BMW,M3
6975,727872,49,64,castanho,ruivo,masculino,P6ITRR,Mercedes-Benz,E-Class
7041,733212,56,55,castanho,castanho,feminino,8L11Q2,GMC,Sierra 1500


In [134]:
df_detran.dtypes

id                 int64
idade              int64
altura             int64
cor_olhos         object
cor_cabelo        object
genero            object
placa             object
marca_veiculo     object
modelo_veiculo    object
dtype: object

In [135]:
df_detran[df_detran["placa"].str.contains("H42W")]

,id,idade,altura,cor_olhos,cor_cabelo,genero,placa,marca_veiculo,modelo_veiculo
915,183779,21,65,azul,loiro,feminino,H42W0X,Fiat,Uno
3529,423327,30,70,castanho,castanho,masculino,0H42W2,Jeep,Renegade
6240,664760,21,71,preto,preto,masculino,4H42WR,Volkswagen,Fusca


In [136]:
suspeitos_placa = df_detran[df_detran["placa"].str.contains("H42W")]

### Passo 4 — cruzando as pistas

In [137]:
# TODO
suspeitos_academia.merge(df_pessoa, left_on = "pessoa_id", right_on = "id", suffixes = ("_esq", "_dir"))

,id_esq,pessoa_id,nome_esq,data_matricula,plano,id_dir,nome_dir,detran_id,numero_endereco,rua,cpf
0,48Z7A,28819,Severino Bezerra Lima,20160305,ouro,28819,Severino Bezerra Lima,173289,111,Fisk Rd,138909730
1,48Z55,67318,Genildo Cavalcanti Farias,20160101,ouro,67318,Genildo Cavalcanti Farias,423327,530,"Washington Pl, Apt 3A",871539279


In [138]:
cruzamento1 = suspeitos_academia.merge(df_pessoa, left_on = "pessoa_id", right_on = "id", suffixes = ("_esq", "_dir"))

In [139]:
suspeitos_placa.merge(df_pessoa, left_on = "id", right_on = "detran_id", suffixes = ("_esq", "_dir"))

,id_esq,idade,altura,cor_olhos,cor_cabelo,genero,placa,marca_veiculo,modelo_veiculo,id_dir,nome,detran_id,numero_endereco,rua,cpf
0,183779,21,65,azul,loiro,feminino,H42W0X,Fiat,Uno,78193,Maxine Whitely,183779,110,Fisk Rd,137882671
1,423327,30,70,castanho,castanho,masculino,0H42W2,Jeep,Renegade,67318,Genildo Cavalcanti Farias,423327,530,"Washington Pl, Apt 3A",871539279
2,664760,21,71,preto,preto,masculino,4H42WR,Volkswagen,Fusca,51739,Tushar Chandra,664760,312,Phi St,137882671


In [140]:
cruzamento2 = suspeitos_placa.merge(df_pessoa, left_on = "id", right_on = "detran_id", suffixes = ("_esq", "_dir"))

In [141]:
cruzamento1[cruzamento1["id_dir"].isin(cruzamento2["id_dir"])]

,id_esq,pessoa_id,nome_esq,data_matricula,plano,id_dir,nome_dir,detran_id,numero_endereco,rua,cpf
1,48Z55,67318,Genildo Cavalcanti Farias,20160101,ouro,67318,Genildo Cavalcanti Farias,423327,530,"Washington Pl, Apt 3A",871539279


In [142]:
cruzamento_final = cruzamento1[cruzamento1["id_dir"].isin(cruzamento2["id_dir"])]

### Passo 5 — confirmar com o check-in

In [143]:
# TODO
df_checkin_academia.sample(10)

,matricula_id,data_checkin,hora_entrada,hora_saida
2151,8I2HW,20170804,473,893
393,4D5R1,20170611,290,736
1199,03487,20180308,176,1111
120,8T75D,20180113,700,952
880,7BQKU,20170810,188,762
1548,344VM,20170723,309,850
755,T2JF2,20170927,773,904
256,51SWX,20170809,1171,1189
1158,CGW3X,20180306,1068,1190
1559,X377L,20180122,875,994


In [144]:
df_checkin_academia.dtypes

matricula_id    object
data_checkin     int64
hora_entrada     int64
hora_saida       int64
dtype: object

In [145]:
df_checkin_academia[(df_checkin_academia["matricula_id"] == "48Z55")]

,matricula_id,data_checkin,hora_entrada,hora_saida
2701,48Z55,20180109,1530,1700


## Fase 3 — Resposta final na Silver

Chegou a hora de publicar sua conclusão como uma tabela — o entregável desta tarefa. Monte um DataFrame de **1 linha** com estas colunas:

| coluna | conteúdo |
|---|---|
| `nome_suspeito` | o nome completo da pessoa em `pessoa` |
| `placa_veiculo` | a placa (de `detran`) que fechou o caso |
| `pista_academia` | qual detalhe da matrícula (início do `id` + status do plano) bateu com o depoimento |
| `pista_veiculo` | qual trecho da placa bateu com o depoimento |
| `justificativa` | 1-2 frases explicando o raciocínio (pode ser texto livre) |

E publique com `df_resposta.to_parquet(f"s3://{BUCKET}/silver/resposta_caso.parquet", storage_options=STORAGE_OPTIONS, index=False)` — depois disso, o **MinIO Console** (http://localhost:9001), em `lakehouse/silver/resposta_caso.parquet`, já mostra o resultado.

In [146]:
# TODO: monte df_resposta (1 linha, colunas da tabela acima) e publique na silver
df_resposta = pd.DataFrame({"nome_suspeito": ["Genildo Cavalcanti Farias"], "placa_veiculo": ["0H42W2"], "pista_academia": ['A matrícula 48Z55 começa com "48Z" e o plano é ouro, como o Josenildo descreveu no depoimento.'], "pista_veiculo": ['A placa 0H42W2 contém o trecho "H42W" citado no depoimento do Josenildo.'], "justificativa": ['O Josenildo Pereira da Rocha viu o assassino sair correndo com uma bolsa da academia (matrícula começando com "48Z", plano ouro) e entrar num carro com placa contendo "H42W". Apenas Genildo Cavalcanti Farias atende às duas pistas ao mesmo tempo, e o check-in de 09/01/2018, das 15h30 às 17h00, confirma o relato da Maria Aparecida Nunes, que o reconheceu da academia.']})
df_resposta.to_parquet(f"s3://{BUCKET}/silver/resposta_caso.parquet", storage_options=STORAGE_OPTIONS, index=False)

---

Terminou? `lakehouse/bronze/` deve ter as 6 tabelas desta tarefa, e `lakehouse/silver/resposta_caso.parquet` deve ter sua conclusão — dá pra confirmar tudo pelo MinIO Console (http://localhost:9001) sem precisar de mais nada.